* Use pybullet to simulate data.
* Components:
    - Visual/Camera data.
    - Proprioceptive data.
    - Force/Torque data. 
* Define a reset function.

In [12]:
import math
import time
from pathlib import Path
from PIL import Image

import numpy as np
import pybullet as p
import pybullet_data

### A1. Hello, physics

In [ ]:
physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
planeID = p.loadURDF("plane.urdf")
r2d2 = p.loadURDF("r2d2.urdf")
for i in range(500):
    p.stepSimulation()
    time.sleep(1./30.)
    if i % 100 == 0:
        r2d2Pos, r2d2Orn = p.getBasePositionAndOrientation(r2d2)
        print(r2d2Pos, r2d2Orn)
p.disconnect()

error: Only one local in-process GUI/GUI_SERVER connection allowed. Use DIRECT connection mode or start a separate GUI physics server (ExampleBrowser, App_SharedMemoryPhysics_GUI, App_SharedMemoryPhysics_VR) and connect over SHARED_MEMORY, UDP or TCP instead.

: 

### A2. Load and inspect a robot arm

In [2]:
physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Vendor = Intel
Renderer = Mesa Intel(R) Graphics (MTL)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started


ven = Intel
Workaround for some crash in the Intel OpenGL driver on Linux/Ubuntu
ven = Intel
Workaround for some crash in the Intel OpenGL driver on Linux/Ubuntu


In [3]:
franka_arm = p.loadURDF("franka_panda/panda.urdf",
                        basePosition=[0,0,0],
                        useFixedBase=True)

for joint_idx in range(p.getNumJoints(bodyUniqueId=franka_arm))[:]:
    info = p.getJointInfo(bodyUniqueId=franka_arm,jointIndex=joint_idx)
    print(joint_idx, info[1].decode("utf-8"), info[12].decode("utf-8"))
    if info[2] == p.JOINT_REVOLUTE:
        print("Type of joint: Revolute\n")
    elif info[2] == p.JOINT_PRISMATIC:
        print("Type of joint: Prismatic\n")
    elif info[2] == p.JOINT_SPHERICAL:
        print("Type of joint: Spherical\n")
    elif info[2] == p.JOINT_PLANAR:
        print("Type of joint: Planar\n")
    elif info[2] == p.JOINT_FIXED:
        print("Type of joint: Fixed\n")
        
PANDA_ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
PANDA_FINGER_JOINTS = [9, 10]
# p.disconnect()


0 panda_joint1 panda_link1
Type of joint: Revolute

1 panda_joint2 panda_link2
Type of joint: Revolute

2 panda_joint3 panda_link3
Type of joint: Revolute

3 panda_joint4 panda_link4
Type of joint: Revolute

4 panda_joint5 panda_link5
Type of joint: Revolute

5 panda_joint6 panda_link6
Type of joint: Revolute

6 panda_joint7 panda_link7
Type of joint: Revolute

7 panda_joint8 panda_link8
Type of joint: Fixed

8 panda_hand_joint panda_hand
Type of joint: Fixed

9 panda_finger_joint1 panda_leftfinger
Type of joint: Prismatic

10 panda_finger_joint2 panda_rightfinger
Type of joint: Prismatic

11 panda_grasptarget_hand panda_grasptarget
Type of joint: Fixed



### A3. Move the arm — joint control

In [ ]:
mode = p.POSITION_CONTROL

##### move a single joint
# p.setJointMotorControl2(
#     bodyUniqueId=franka_arm,
#     jointIndex=3,
#     controlMode=mode,
#     targetPosition = 4.0,
#     force = 10
# )

##### move an array of joints.
p.setJointMotorControlArray(
    bodyIndex=franka_arm,
    jointIndices=PANDA_ARM_JOINTS,
    controlMode=mode,
    targetPositions = [0.3]*len(PANDA_ARM_JOINTS),
    forces = [0.003]*len(PANDA_ARM_JOINTS)
)

for i in range(100):
    p.stepSimulation()

### A4. Move the end-effector — inverse kinematics

* getJointState(bodyUniqueId,jointIndex)[0] gives the position. It gives the angle in radians, that's why it is 1 degree of freedom.
* getLinkState(...)[4], gives you the link's pose in world Cartesian space, that's (x,y,z).
* IMPORTANT: Every time you run the line "p.loadURDF" a robotic arm is loaded (on top of each other). Creating multiple collisions and contacts.


In [100]:
p.disconnect()
physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
franka_arm = p.loadURDF("franka_panda/panda.urdf",
                        basePosition=[0,0,0],
                        useFixedBase=True)

for joint_idx in range(p.getNumJoints(bodyUniqueId=franka_arm))[:]:
    info = p.getJointInfo(bodyUniqueId=franka_arm,jointIndex=joint_idx)
    print(joint_idx, info[1].decode("utf-8"), info[12].decode("utf-8"))

PANDA_ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
PANDA_FINGER_JOINTS = [9, 10]

numActiveThreads = 0
stopping threads
destroy semaphore
semaphore destroyed
Thread with taskId 0 exiting
Thread TERMINATED
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
destroy semaphore
semaphore destroyed
Thread with taskId 0 exiting
Thread TERMINATED
destroy main semaphore
main semaphore destroyed
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Ve

In [101]:
# find the end effector: Hand
end_effector_index = None

for i in range(p.getNumJoints(franka_arm)):
    link_name = p.getJointInfo(franka_arm, i)[12].decode("utf-8")

    if link_name == "panda_hand":
        end_effector_index = i
        break

print("End effector index:", end_effector_index)

End effector index: 8


In [103]:
# get number of bodies. It should be 1.
print("Number of bodies in world:", p.getNumBodies())
for i in range(p.getNumBodies()):
    print(i, p.getBodyInfo(i))

# reset position
home_positions = [0, -0.785, 0, -2.356, 0, 1.571, 0.785]
finger_home = [0.04, 0.04]  # open gripper

for j, angle in zip(PANDA_ARM_JOINTS, home_positions):
    p.resetJointState(franka_arm, j, angle, targetVelocity=0)

for j, angle in zip(PANDA_FINGER_JOINTS, finger_home):
    p.resetJointState(franka_arm, j, angle, targetVelocity=0)

p.stepSimulation()

# get the current position and set the target position
state = p.getLinkState(bodyUniqueId=franka_arm, linkIndex=end_effector_index)

print("Position:", state[4])
print("Orientation:", state[5])

move_position = 0.20 # 10cm.
target_pos = np.array([state[4][0] + move_position, state[4][1], state[4][2]])

# check for self-collision 
contacts = p.getContactPoints(bodyA=franka_arm)
print("Contacts after reset:", len(contacts))
for c in contacts:
    print("  bodyA link:", c[3], " bodyB:", c[2], " bodyB link:", c[4])
    print(f"linkA={c[3]} linkB={c[4]} distance={c[8]:.6f} normalForce={c[9]:.6f}")

# calculate Inverse Kinematics.
joint_angles = p.calculateInverseKinematics(
    bodyUniqueId=franka_arm,
    endEffectorLinkIndex=end_effector_index,
    targetPosition=target_pos,
    maxNumIterations=200,
    residualThreshold=1e-7
)

print(f"target_pos:{target_pos}")

p.setJointMotorControlArray(
    bodyIndex=franka_arm,
    jointIndices=PANDA_ARM_JOINTS + PANDA_FINGER_JOINTS,
    controlMode=p.POSITION_CONTROL,
    targetPositions = joint_angles,
    forces = [100]*len(PANDA_ARM_JOINTS + PANDA_FINGER_JOINTS)
)

for _ in range(700):
    p.stepSimulation()

print(f"Position-Link State after IK: {p.getLinkState(bodyUniqueId=franka_arm, linkIndex=end_effector_index)[4]}")
print(f"\nDifference between Link Position after IK-solver and target position: {np.array(p.getLinkState(bodyUniqueId=franka_arm, linkIndex=end_effector_index)[4]) - np.array(target_pos)}")
print(np.isclose(np.array(p.getLinkState(bodyUniqueId=franka_arm, linkIndex=end_effector_index)[4]), np.array(target_pos), rtol=1e-05, atol=1e-08))
print(np.isclose(np.array(p.getLinkState(bodyUniqueId=franka_arm, linkIndex=end_effector_index)[4]), np.array(target_pos), rtol=1e-05, atol=1e-08).all())
### check the Panda Arm Joints reached the IK solution.
current_arm_positions = [p.getJointState(franka_arm, j)[0] for j in PANDA_ARM_JOINTS]
print("Commanded arm joint targets:", joint_angles[:7])
print("Actual arm joint positions:", current_arm_positions)

Number of bodies in world: 1
0 (b'panda_link0', b'panda')
Position: (0.30701956152915955, -5.221265694105304e-12, 0.5902695655822754)
Orientation: (1.0, 0.00019908169633708894, -6.818621578144222e-16, 3.4637366759515675e-12)
Contacts after reset: 0
target_pos:[ 5.07019562e-01 -5.22126569e-12  5.90269566e-01]
Position-Link State after IK: (0.5070195198059082, -5.4072302191343624e-12, 0.5902695655822754)

Difference between Link Position after IK-solver and target position: [-4.17232513e-08 -1.85964525e-13  0.00000000e+00]
[ True  True  True]
True
Commanded arm joint targets: (-1.4430573373499742e-08, -0.2308713963938028, -1.882984919161436e-08, -1.9646084296463802, 2.5362338600787615e-07, 1.9248252619550679, 0.7849999992938285)
Actual arm joint positions: [-1.4430574896394679e-08, -0.23087139639380272, -1.8829849191614346e-08, -1.9646084296463797, 2.53623385867588e-07, 1.9248252619550683, 0.7849999992938282]


### A5. Build a minimal `reset()` / `step()` environment

In [ ]:
p.disconnect()
physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
planeID = p.loadURDF("plane.urdf")
franka_arm = p.loadURDF("franka_panda/panda.urdf",
                        basePosition=[0,0,0],
                        useFixedBase=True)

for joint_idx in range(p.getNumJoints(bodyUniqueId=franka_arm))[:]:
    info = p.getJointInfo(bodyUniqueId=franka_arm,jointIndex=joint_idx)
    print(joint_idx, info[1].decode("utf-8"), info[12].decode("utf-8"))

PANDA_ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
PANDA_FINGER_JOINTS = [9, 10]



# this enables contact with the table.
for j in PANDA_ARM_JOINTS:
    p.enableJointForceTorqueSensor(
        franka_arm,
        j,
        True
    )

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Ve

In [116]:
class MiniArmEnv():
    def __init__(self, robotId):
        self.robotId = robotId
        self.PANDA_ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
        self.PANDA_FINGER_JOINTS = [9, 10]
        self.end_effector = self._get_end_effector()
        self.n_step_simulation = 800
        self.physics_engine_update = 240 # 240 Hz.
        self.controller_update = 60 # 60 Hz
        self.reset()
    
    def reset(self):
        home_positions = [0, -0.785, 0, -2.356, 0, 1.571, 0.785]
        finger_home = [0.04, 0.04]  # open gripper

        for j, angle in zip(self.PANDA_ARM_JOINTS, home_positions):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        for j, angle in zip(self.PANDA_FINGER_JOINTS, finger_home):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        p.stepSimulation()
    
    def step(self, action):
        self.action = action

        # inverse kinematics
        joint_angles = p.calculateInverseKinematics(
            bodyUniqueId=self.robotId,
            endEffectorLinkIndex=self.end_effector,
            targetPosition=self.action,
            maxNumIterations=300,
            residualThreshold=1e-7
        )

        # move the arm
        p.setJointMotorControlArray(
            bodyIndex=self.robotId,
            jointIndices=self.PANDA_ARM_JOINTS + self.PANDA_FINGER_JOINTS,
            controlMode=p.POSITION_CONTROL,
            targetPositions = joint_angles,
            forces = [500]*len(self.PANDA_ARM_JOINTS + self.PANDA_FINGER_JOINTS)
        )

        p.setTimeStep(self.physics_engine_update / self.controller_update)

        for _ in range(self.n_step_simulation):
            p.stepSimulation()

        # check the target position was reached
        self.reached_action = p.getLinkState(self.robotId, self.end_effector)[4]
        if self._check_target_position():
            print("Reached action matches defined action")

        obs = self._get_obs()
        reward = None
        done = None
        info = None

        return obs, reward, done, info

    def _get_end_effector(self):
        end_effector_index = None

        for i in range(p.getNumJoints(self.robotId)):
            link_name = p.getJointInfo(self.robotId, i)[12].decode("utf-8")

            if link_name == "panda_hand":
                end_effector_index = i
                break
        return end_effector_index
    
    def _get_obs(self):
        position = p.getLinkState(self.robotId, self.end_effector)[4]
        return {
            "position": position
        }

    def _check_target_position(self):
        return np.isclose(np.array(self.reached_action), self.action, rtol=1e-05, atol=1e-08).all()
    


In [82]:
arm_env = MiniArmEnv(robotId=franka_arm)
arm_env.step(action=np.array([1,10,2.7]))

({'position': (0.25974854826927185, 0.7747482657432556, 0.11558958888053894)},
 None,
 None,
 None)

### A6. Add a camera and read pixels


In [123]:
p.disconnect()
physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
planeID = p.loadURDF("plane.urdf")

franka_arm = p.loadURDF("franka_panda/panda.urdf",
                        basePosition=[0,0,0],
                        useFixedBase=True)


numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Ve

In [ ]:
view_matrix = p.computeViewMatrix(cameraEyePosition=[1,0,2], 
                                  cameraTargetPosition=[0,0,1],
                                  cameraUpVector=[0,1,0.5]
                                  )
projection_matrix = p.computeProjectionMatrixFOV(fov=60,
                                              aspect=1.0,
                                              nearVal=0.1,
                                              farVal=10
                                             )
                                
width, height, rgb, depth, segmentation = p.getCameraImage(width=1200,height=1200,viewMatrix=view_matrix,projectionMatrix=projection_matrix)

# Convert RGB data to a NumPy array
rgb_array = np.array(rgb, dtype=np.uint8).reshape(height, width, 4)

# Remove alpha channel (RGBA -> RGB)
rgb_array = rgb_array[:, :, :3]

# Save image
image = Image.fromarray(rgb_array)
image.save("camera_view.png")

: 

### A7. Force/torque sensing

* the force applied when z=0 is very soft, but there is a difference between z=0.03 and z=0.0 which means that there is a contact happening.

In [ ]:
arm_env = MiniArmEnv(robotId=franka_arm)
PANDA_ARM_JOINTS = arm_env.PANDA_ARM_JOINTS
PANDA_FINGER_JOINTS = arm_env.PANDA_FINGER_JOINTS

# move down slowly
arm_env.step(np.array([0.1, 0, -1.5]))


states = p.getJointStates(
    bodyUniqueId=franka_arm,
    jointIndices=PANDA_ARM_JOINTS
)

for j, s in zip(PANDA_ARM_JOINTS, states):
    print(f"Joint {j}")
    print("joint_reaction_forces:", s[2])
    print(" joint_motor_torque:", s[3])

print(
    p.getLinkState(
        franka_arm,
        arm_env.end_effector
    )[4]
)

Joint 0
joint_reaction_forces: (0.011453683422313725, 0.00957629365316153, -0.021596890614145152, 0.0036953116975611067, 0.013483534170008846, -0.000458147336892549)
 joint_motor_torque: 0.010315021234951618
Joint 1
joint_reaction_forces: (0.01790940241406669, -0.016602767695529756, 0.009576293650752384, -0.011873202839159394, -0.0032725721331504066, -0.0007163760965626675)
 joint_motor_torque: 0.01374689638093253
Joint 2
joint_reaction_forces: (0.013534842450565878, 0.014999345863185224, 0.016680582904043083, -0.011125152024702253, 0.004890940626719794, -1.4645034126193485e-05)
 joint_motor_torque: 0.0022050099334440625
Joint 3
joint_reaction_forces: (0.010964198748679998, 0.018340879031977966, -0.014599697359147207, -0.00945465694494077, -0.0001850765898557194, 0.0008791523334197395)
 joint_motor_torque: -0.00542992116202243
Joint 4
joint_reaction_forces: (0.007987771964050329, 0.015351855280216714, 0.018063708772399354, -0.007372481699535694, 8.299093078399377e-05, 0.000319510878562

In [118]:
contacts = p.getContactPoints(
    bodyA=franka_arm,
    bodyB=planeID
)

for c in contacts:
    print("Link index:", c[3])
    print("Contact position on robot:", c[5])
    print("Contact normal force:", c[9])
    print("---")

Link index: 10
Contact position on robot: (0.7950696904494954, -0.2525178064918681, -0.00390999957515192)
Contact normal force: 0.008958935007736467
---
Link index: 9
Contact position on robot: (0.8053094253437223, -0.22556583335618854, -0.005926975092645968)
Contact normal force: 0.004167482662373326
---
Link index: 8
Contact position on robot: (0.8660052980377709, 0.0265698464282854, 0.001415419447060133)
Contact normal force: 0.006467830113931698
---
